In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from sklearn.metrics import make_scorer
from sklearn.preprocessing import LabelEncoder,StandardScaler
from xgboost import plot_importance
# from pandas_profiling import ProfileReport

In [ ]:
trainData = pd.read_csv('train.csv')
# trainData = trainData.drop(columns='Unnamed: 0')


testData = pd.read_csv('test.csv')
# testData = testData.drop(columns='Unnamed: 0')

In [ ]:
def mcrmse_loss(y_true, y_pred, N=3):
    """
    Calculates competition eval metric
    """
    assert len(y_true) == len(y_pred)
    n = len(y_true)
    return np.sum(np.sqrt(np.sum((y_true - y_pred)**2, axis=0)/n)) / N

In [ ]:
le = LabelEncoder()

In [ ]:
trainData[['sequence', 'structure', 'predicted_loop_type',
    'b1_sequence', 'a1_sequence', 'b1_structure', 'a1_structure',
    'b1_predicted_loop_type', 'a1_predicted_loop_type', 'b2_sequence',
    'a2_sequence', 'b2_structure', 'a2_structure',
    'b2_predicted_loop_type', 'a2_predicted_loop_type', 'b3_sequence',
    'a3_sequence', 'b3_structure', 'a3_structure',
    'b3_predicted_loop_type', 'a3_predicted_loop_type', 'b4_sequence',
    'a4_sequence', 'b4_structure', 'a4_structure',
    'b4_predicted_loop_type', 'a4_predicted_loop_type', 'b5_sequence',
    'a5_sequence', 'b5_structure', 'a5_structure',
    'b5_predicted_loop_type', 'a5_predicted_loop_type']]=trainData[['sequence', 'structure', 'predicted_loop_type',
    'b1_sequence', 'a1_sequence', 'b1_structure', 'a1_structure',
    'b1_predicted_loop_type', 'a1_predicted_loop_type', 'b2_sequence',
    'a2_sequence', 'b2_structure', 'a2_structure',
    'b2_predicted_loop_type', 'a2_predicted_loop_type', 'b3_sequence',
    'a3_sequence', 'b3_structure', 'a3_structure',
    'b3_predicted_loop_type', 'a3_predicted_loop_type', 'b4_sequence',
    'a4_sequence', 'b4_structure', 'a4_structure',
    'b4_predicted_loop_type', 'a4_predicted_loop_type', 'b5_sequence',
    'a5_sequence', 'b5_structure', 'a5_structure',
    'b5_predicted_loop_type', 'a5_predicted_loop_type']].apply(le.fit_transform)


In [ ]:
testData[['sequence', 'structure', 'predicted_loop_type',
    'b1_sequence', 'a1_sequence', 'b1_structure', 'a1_structure',
    'b1_predicted_loop_type', 'a1_predicted_loop_type', 'b2_sequence',
    'a2_sequence', 'b2_structure', 'a2_structure',
    'b2_predicted_loop_type', 'a2_predicted_loop_type', 'b3_sequence',
    'a3_sequence', 'b3_structure', 'a3_structure',
    'b3_predicted_loop_type', 'a3_predicted_loop_type', 'b4_sequence',
    'a4_sequence', 'b4_structure', 'a4_structure',
    'b4_predicted_loop_type', 'a4_predicted_loop_type', 'b5_sequence',
    'a5_sequence', 'b5_structure', 'a5_structure',
    'b5_predicted_loop_type', 'a5_predicted_loop_type']]=testData[['sequence', 'structure', 'predicted_loop_type',
    'b1_sequence', 'a1_sequence', 'b1_structure', 'a1_structure',
    'b1_predicted_loop_type', 'a1_predicted_loop_type', 'b2_sequence',
    'a2_sequence', 'b2_structure', 'a2_structure',
    'b2_predicted_loop_type', 'a2_predicted_loop_type', 'b3_sequence',
    'a3_sequence', 'b3_structure', 'a3_structure',
    'b3_predicted_loop_type', 'a3_predicted_loop_type', 'b4_sequence',
    'a4_sequence', 'b4_structure', 'a4_structure',
    'b4_predicted_loop_type', 'a4_predicted_loop_type', 'b5_sequence',
    'a5_sequence', 'b5_structure', 'a5_structure',
    'b5_predicted_loop_type', 'a5_predicted_loop_type']].apply(le.fit_transform)

In [ ]:
trainData.drop(['id_seqpos','id'],axis=1,inplace=True)
testData.drop(['id_seqpos','id'],axis=1,inplace=True)

In [ ]:
X = trainData.drop(['reactivity', 'deg_Mg_pH10', 'deg_Mg_50C'], axis=1)
y = trainData[['reactivity', 'deg_Mg_pH10', 'deg_Mg_50C']]

In [ ]:
# scaler = StandardScaler()
# X = scaler.fit_transform(X)
# testData = scaler.transform(testData)

In [ ]:
# scaler = MinMaxScaler()
# X = scaler.fit_transform(X)
# X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from xgboost import XGBRegressor
xgb_model = XGBRegressor(max_depth=6,learning_rate=0.1,colsample_bytree=0.9,reg_alpha=5,reg_lambda=6
                      ,min_child_weight = 2,n_estimators=1500)

xgb_model.fit(X_train, y_train)

In [ ]:
y_pred_xgb =xgb_model.predict(X_test)

In [ ]:
mcrmse_loss(y_test,y_pred_xgb)

In [ ]:
test_id = pd.read_csv('test.csv')
K = pd.DataFrame(y_pred_xgb,columns = ['reactivity', 'deg_Mg_pH10', 'deg_Mg_50C'])
test_id = test_id['id_seqpos']
K2 = pd.concat([test_id,K],axis = 1)
K2.to_csv('result_new_reg6.csv',index = False)

# Error analysis

In [ ]:
# Create a DataFrame to store the output
error_analysis = pd.DataFrame()

error_analysis['reactivity_actual'] = y_test['reactivity']
error_analysis['reactivity_predicted'] = y_pred_xgb[:, 0]

error_analysis['deg_Mg_pH10_actual'] = y_test['deg_Mg_pH10']
error_analysis['deg_Mg_pH10_predicted'] = y_pred_xgb[:, 1]


error_analysis['deg_Mg_50C_actual'] = y_test['deg_Mg_50C']
error_analysis['deg_Mg_50C_predicted'] = y_pred_xgb[:, 2]

error_analysis

In [ ]:
error_analysis = pd.concat([error_analysis, X_test], axis=1)

In [ ]:
correlation_matrix = error_analysis[['reactivity_actual', 'reactivity_predicted', 'deg_Mg_pH10_actual', 'deg_Mg_pH10_predicted', 'deg_Mg_50C_actual', 'deg_Mg_50C_predicted']].corr()

# Create a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Plot feature importances
plt.figure(figsize=(100,100))
plot_importance(xgb_model, max_num_features=40)  # You can change max_num_features to display more or fewer features
plt.title("Feature Importance")
plt.show()